In [ ]:
# -*- coding: utf-8 -*-
"""
Reddit Comment Labeling Pipeline (LM Studio, OpenAI-compatible API)
-------------------------------------------------------------------
- Input: List[str] where each item is a raw Reddit comment text.
- Retries: Up to 3 attempts per (comment, task) with exponential backoff.
- Strict JSON validation against task schemas; otherwise retry.
- Output: NDJSON (one line per (comment_index, task)).
- Tasks implemented:
    - stance_intensity
    - epistemic_modality
    - justification_density
    - responsiveness (will ABSTAIN because no parent text provided)
    - civility

Requirements:
    pip install requests
    reddit data has to be stored locally in a folder called 'data'.

LM Studio:
    - Enable the local HTTP server in LM Studio (OpenAI-compatible API).
    - Default endpoint: http://localhost:1234/v1
    - Set MODEL_NAME to the exact local model identifier shown in LM Studio.

"""

import json
import time
from typing import Dict, Any, List, Optional
import requests

# ------------------------
# Configuration
# ------------------------

LMSTUDIO_BASE_URL = "http://localhost:1234/v1"   # Change if LM Studio runs on a different host/port
MODEL_NAME = "your-local-model-name-here"        # e.g., "qwen2.5-7b-instruct" or model name shown in LM Studio
TIMEOUT_SECONDS = 60                              # HTTP request timeout
MAX_RETRIES = 3                                   # Max attempts per (comment, task)
RETRY_BACKOFF_SECONDS = 1.5                       # Exponential backoff base

# ------------------------
# Task schema specifications (for strict validation)
# ------------------------

TASK_SPECS: Dict[str, Dict[str, Any]] = {
    "stance_intensity": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "stance_intensity",
        "score_key": "score",
        "score_type": (int, float, str),   # "ABSTAIN" allowed
        "confidence_range": (0.0, 1.0),
        "score_range": (-1.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "epistemic_modality": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "epistemic_modality",
        "score_key": "score",
        "score_type": (int, float, str),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "justification_density": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "justification_density",
        "score_key": "score",
        "score_type": (int, float, str),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, float("inf")),  # non-negative float
        "allow_abstain": True,
        "label_key": None,
    },
    "responsiveness": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "responsiveness",
        "score_key": "score",
        "score_type": (int, float, str),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,  # Will ABSTAIN because no parent text is provided
        "label_key": None,
    },
    "civility": {
        "schema_keys": {"task", "label", "confidence"},
        "task_value": "civility",
        "score_key": None,
        "label_key": "label",
        "label_type": (int, str),  # 1..6 or "ABSTAIN"
        "confidence_range": (0.0, 1.0),
        "allow_abstain": True,
    },
}

# ------------------------
# System prompt (passed verbatim to the model)
# ------------------------

SYSTEM_PROMPT = """You are a careful, literal discourse annotator. Use ONLY the provided TEXT,
PARENT_TEXT (if given), and TOPIC_DEF. Do not infer beyond the text or
use any external knowledge. If evidence is insufficient, return "ABSTAIN".
Output must be strictly valid JSON matching the registered schema for the task.
Include a "confidence" field in [0,1].
B.2 B2. Instruction templates and schemas
Below we include representative instruction + schema pairs. All tasks follow the same schema:
English only; if ¡15 tokens then ABSTAIN.
Stance intensity (continuous; [−1, 1])
TASK: Estimate the author’s stance toward TOPIC_DEF as a continuous
score in [-1.0, 1.0]. Use explicit valenced statements linked to the topic.
If ambiguous or absent, return "ABSTAIN".
JSON schema:
{
"task": "stance_intensity",
"score": <float in [-1.0,1.0] or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Epistemic modality (continuous; [0,1])
TASK: Score epistemic modality: density of hedges, modal verbs, and explicit
uncertainty, normalized by text length. Higher = more expressed humility.
Return "ABSTAIN" if <15 tokens.
JSON schema:
{
"task": "epistemic_modality",
"score": <float in [0.0,1.0] or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Justification density (per 100 words)
TASK: Identify distinct justification units (claim + warrant) and report
the count normalized per 100 words. Use span-unitization rules provided.
Return "ABSTAIN" if insufficient evidence.
JSON schema:
{
"task": "justification_density",
"score": <non-negative float or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Responsiveness (continuous; [0,1])
TASK: Rate how directly this reply addresses its PARENT_TEXT’s content
(semantic overlap / engagement). 0 = unrelated; 1 = direct, on-topic response.
Return "ABSTAIN" if PARENT_TEXT not provided.
JSON schema:
{
"task": "responsiveness",
"score": <float in [0.0,1.0] or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Civility (ordinal 1–6)
TASK: Rate civility on a 1 (highly uncivil/insulting) to 6 (highly civil)
scale, focusing on tone, directness, and presence of insults.
Return "ABSTAIN" if ambiguous.
JSON schema:
{
"task": "civility",
"label": <integer 1..6 or "ABSTAIN">,
"confidence": <float in [0,1]>,
"probs": { "1":.., "2":.., ... "6":.. } // optional
}
"""

# Template that tells the model which single task to execute now
TASK_INSTRUCTION_TEMPLATE = (
    "Now perform ONLY the task = {task_name}. "
    "Return strictly valid JSON for that task and nothing else (no Markdown). "
    "Ensure keys and value ranges match the schema exactly."
)

# ------------------------
# HTTP call utilities
# ------------------------

def call_lmstudio_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """
    Call LM Studio (OpenAI-compatible) Chat Completions API and return raw text.
    Raises requests.RequestException on network/HTTP errors.
    """
    url = f"{LMSTUDIO_BASE_URL}/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECONDS)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"].strip()

def _is_number(x: Any) -> bool:
    """Helper: check numeric types."""
    return isinstance(x, (int, float))

# ------------------------
# Response validation
# ------------------------

def validate_response(task: str, obj: Dict[str, Any]) -> Optional[str]:
    """
    Validate the model's JSON against the task schema.
    Returns None if valid; otherwise returns an error message.
    """
    spec = TASK_SPECS[task]

    # Must be a dict
    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    # Required keys present?
    missing = spec["schema_keys"] - set(obj.keys())
    if missing:
        return f"Missing required keys: {sorted(missing)}"

    # Exact "task" value
    if obj.get("task") != spec["task_value"]:
        return f'Field "task" must be "{spec["task_value"]}"'

    # confidence in [0,1]
    conf = obj.get("confidence")
    if not _is_number(conf):
        return '"confidence" must be a number'
    lo, hi = spec["confidence_range"]
    if not (lo <= conf <= hi):
        return f'"confidence" must be in [{lo}, {hi}]'

    # Civility uses an ordinal "label"
    if task == "civility":
        label = obj.get(spec["label_key"])
        if isinstance(label, str):
            if label != "ABSTAIN" and not label.isdigit():
                return '"label" must be integer 1..6 or "ABSTAIN"'
        elif isinstance(label, int):
            if not (1 <= label <= 6):
                return '"label" integer must be in [1,6]'
        else:
            return '"label" must be int or "ABSTAIN"'
        return None

    # All other tasks use a numeric "score" or "ABSTAIN"
    score = obj.get(spec["score_key"])
    if isinstance(score, str):
        if not spec["allow_abstain"] or score != "ABSTAIN":
            return f'"{spec["score_key"]}" must be a number or "ABSTAIN"'
        return None
    elif _is_number(score):
        lo, hi = spec["score_range"]
        if not (lo <= float(score) <= hi):
            return f'"{spec["score_key"]}" out of range [{lo}, {hi}]'
        return None
    else:
        return f'"{spec["score_key"]}" must be number or "ABSTAIN"'

# ------------------------
# Prompt construction
# ------------------------

def build_user_prompt(task: str, text: str) -> str:
    """
    Compose the user message that includes only TEXT and the task directive.
    Note: We do not provide PARENT_TEXT or TOPIC_DEF in this simplified setup.
    """
    parts = {
        "TEXT": text
    }
    directive = TASK_INSTRUCTION_TEMPLATE.format(task_name=task)
    # The model receives a compact JSON blob for content, followed by the directive.
    return json.dumps(parts, ensure_ascii=False) + "\n\n" + directive

# ------------------------
# Single annotation with retries
# ------------------------

def annotate_one(task: str, text: str) -> Dict[str, Any]:
    """
    Run one (task, text) through LM Studio with retries and strict JSON validation.
    Returns the valid task-JSON on success, or {"task": task, "error": "..."} after all retries fail.
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(task, text)},
            ]
            raw = call_lmstudio_chat(messages, temperature=0.0)

            # Expect strictly JSON (no Markdown). Try to parse.
            obj = json.loads(raw)

            # Validate against the schema for this task
            err = validate_response(task, obj)
            if err is None:
                return obj
            else:
                last_err = f"Schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            # Network or HTTP-level error
            last_err = f"HTTP error (attempt {attempt}): {e}"

        except json.JSONDecodeError as e:
            # Model returned non-JSON or malformed JSON
            last_err = f"JSON parse error (attempt {attempt}): {e}"

        # Exponential backoff before retrying
        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    # After exhausting all retries, return an error object
    return {
        "task": task,
        "error": last_err or "Unknown error",
    }

# ------------------------
# NDJSON writer
# ------------------------

def write_ndjson_line(fp, obj: Dict[str, Any]) -> None:
    """Write a single JSON object as one NDJSON line."""
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")

# ------------------------
# Main pipeline
# ------------------------

def run_pipeline(
        comments: List[str],
        tasks: Optional[List[str]] = None,
        ndjson_path: str = "labels.ndjson",
) -> None:
    """
    Run the labeling pipeline.

    Args:
        comments: List of raw comment texts (List[str]).
        tasks: Optional list of task names; defaults to all implemented tasks.
        ndjson_path: Output file path for NDJSON.

    Output format (NDJSON):
        One line per (comment_index, task), with a JSON object:
        {
          "comment_index": <int>,           # index of the comment in the input list
          "task": "<task_name>",
          "result": { ... }                 # validated task JSON or {"task": "<task>", "error": "..."}
        }
    """
    if tasks is None:
        tasks = list(TASK_SPECS.keys())

    with open(ndjson_path, "w", encoding="utf-8") as f:
        for idx, text in enumerate(comments):
            for task in tasks:
                result = annotate_one(task, text)
                out = {
                    "comment_index": idx,
                    "task": task,
                    "result": result,
                }
                write_ndjson_line(f, out)

# ------------------------
# Example usage
# ------------------------

if __name__ == "__main__":
    # Minimal demo comments list (replace with your actual comments)
    demo_comments: List[str] = [
        "I don't think that policy will work; maybe we should gather more data first.",
        "This is dumb. You clearly have no idea what you're talking about.",
        "Interesting point—I'm not sure though, it could depend on regional differences and the sample size.",
    ]

    # Choose tasks (or set to None for all)
    tasks = [
        "stance_intensity",
        "epistemic_modality",
        "justification_density",
        "responsiveness",  # Will ABSTAIN (no parent text provided)
        "civility",
    ]

    run_pipeline(demo_comments, tasks=tasks, ndjson_path="labels.ndjson")
    print("Done. Wrote NDJSON to labels.ndjson")
